# GloVe (2014)

From-scratch co-occurrence counts, embeddings, weighted loss, and training.

Run setup and definitions, then the benchmark cell. Training is opt-in in the final cell.
On Neovim/Colab, mount Drive using `colab drivemount --session eos` before checkpointed training.


In [ ]:
# Imports and device (seed is configured in cell 5) import torch
import random
import torch
import torch.nn as nn
from collections import Counter
from torch.utils.data import DataLoader, TensorDataset

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else ("mps" if torch.backends.mps.is_available() else "cpu")
)
print(f"Using device: {device}")

In [ ]:
# eos-data: text8 (auto-downloads on Colab, reuses the local copy on Mac)
import os

try:
    import google.colab
    ON_COLAB = True
except ImportError:
    ON_COLAB = False

if ON_COLAB:
    DATA_DIR = "/root/colab-data"
    DATA_PATH = os.path.join(DATA_DIR, "text8")
    if not os.path.exists(DATA_PATH):
        os.makedirs(DATA_DIR, exist_ok=True)
        import shutil
        import urllib.request
        import zipfile
        print("downloading text8...")
        tmp_extract = DATA_DIR + ".tmp-extract"
        shutil.rmtree(tmp_extract, ignore_errors=True)
        os.makedirs(tmp_extract)
        try:
            urllib.request.urlretrieve(
                "http://mattmahoney.net/dc/text8.zip",
                os.path.join(tmp_extract, "text8.zip"),
            )
            with zipfile.ZipFile(os.path.join(tmp_extract, "text8.zip")) as archive:
                archive.extractall(tmp_extract)
            os.rename(os.path.join(tmp_extract, "text8"), DATA_PATH)
        finally:
            shutil.rmtree(tmp_extract, ignore_errors=True)
else:
    DATA_PATH = "data/text8"

print("data:", DATA_PATH, "exists:", os.path.exists(DATA_PATH))

In [ ]:
# Load corpus and build vocabulary (ID 0 is reserved for padding)
raw_tokens = open(DATA_PATH).read().split()

tokens_counter = Counter(raw_tokens)
word_to_idx = {token: i+1 for i, token in enumerate(tokens_counter.keys())}
idx_to_word = {i+1: token for i, token in enumerate(tokens_counter.keys())}
print(f"Total Tokens: {len(raw_tokens)}")
print(f"Vocab Size: {len(word_to_idx)}")

token_ids = [word_to_idx.get(token) for token in raw_tokens]
assert len(token_ids) == len(raw_tokens)

In [ ]:
# Configuration - rerun this cell after changing parameters.
from pathlib import Path

# Reproducibility (applied before model initialization / shuffling)
seed = 42
torch.manual_seed(seed)
random.seed(seed)

# Data and co-occurrence (DATA_PATH is initialized in the earlier data-setup cell)
N = len(token_ids)
sample_size = 10_000
C = 1
chunk_size = 100_000
count_weighting = "unit"  # Current implementation uses unweighted counts.

# Model
vocab_size = len(word_to_idx)
embedding_dim = 50

# Training - these are for a new run, not the completed 50-epoch checkpoint.
batch_size = 32
learning_rate = 0.01
epochs = 5
optimizer_name = "SGD"  # Only SGD is implemented in the current loops.

# GloVe loss weighting
count_threshold = 100
alpha = 0.75

# Benchmark
benchmark_sample_sizes = (10_000, 100_000, 300_000)
benchmark_large_sample_sizes = (1_000_000,)
benchmark_steps = 100
benchmark_warmup = 10
benchmark_time_limit_seconds = 30
benchmark_min_available_ram_gib = 2

# Explicit execution switches (preserved from the existing cells)
RUN_FULL_COUNTS = False
RUN_TRAINING = False

# Artifacts - Drive must already be mounted on Colab.
artifact_dir = Path("/content/drive/MyDrive/genesis-lab/glove" if ON_COLAB else "outputs/glove")
checkpoint_path = str(artifact_dir / "checkpoint.pt")  # New-run output
full_counts_path = artifact_dir / f"text8-full-counts-C{C}-{count_weighting}.pt"
trained_run_id = "20261013-022058"
trained_checkpoint_path = artifact_dir / "runs" / trained_run_id / "checkpoint.pt"
# The local trained checkpoint path requires downloading the Drive file first.

# Metadata saved with new checkpoints
run_config = dict(
    seed=seed, sample_size=sample_size, C=C, chunk_size=chunk_size,
    vocab_size=vocab_size, embedding_dim=embedding_dim,
    learning_rate=learning_rate, epochs=epochs, batch_size=batch_size,
    count_threshold=count_threshold, alpha=alpha, optimizer=optimizer_name,
    weighting=count_weighting,
)

In [ ]:
def iter_pair_counts(token_ids, C, chunk_size):
    assert C >= 1
    assert chunk_size >= 1
    N = len(token_ids)
    if N == 0:
        return

    token_ids_expanded = [0]*C + token_ids + [0]*C
    token_ids_expanded_tensor = torch.tensor(token_ids_expanded, dtype=torch.int32)
    assert token_ids_expanded_tensor.shape == (N + 2*C,)
    windows = token_ids_expanded_tensor.unfold(0, 2*C + 1, 1)
    assert windows.shape == (N, 2*C + 1)
    targets = token_ids_expanded_tensor[C:C + N]
    assert targets.shape == (N,)

    for start in range(0, N, chunk_size):
        end = min(start + chunk_size, N)
        B = end - start
        window_chunk = windows[start:end]
        assert window_chunk.shape == (B, 2*C + 1)
        target_chunk = targets[start:end]
        assert target_chunk.shape == (B,)

        left_part = window_chunk[:, :C]
        assert left_part.shape == (B, C)
        right_part = window_chunk[:, C+1:]
        assert right_part.shape == (B, C)
        context = torch.cat([left_part, right_part], dim=1)
        assert context.shape == (B, 2*C)
        target_chunk = target_chunk.unsqueeze(1)
        assert target_chunk.shape == (B, 1)
        target_chunk = target_chunk.expand(B, 2*C)
        assert target_chunk.shape == (B, 2*C)
        result = torch.stack([target_chunk, context], dim=2)
        assert result.shape == (B, 2*C, 2)
        pairs = result.reshape(-1, 2)
        assert pairs.shape == (B*2*C, 2)
        mask = pairs[:, 1] != 0
        assert mask.shape == (B*2*C,)
        filtered_pairs = pairs[mask]
        assert filtered_pairs.shape == (int(mask.sum()), 2)
        unique_pairs, counts = torch.unique(filtered_pairs, dim=0, return_counts=True)
        assert unique_pairs.shape == (counts.shape[0], 2)
        chunk_counts = torch.cat([unique_pairs, counts.unsqueeze(1)], dim=1)
        assert chunk_counts.shape == (unique_pairs.shape[0], 3)

        # Counts are local to this chunk; combine merges them.
        yield chunk_counts



In [ ]:
def combine(token_ids, C, chunk_size, progress=False):
    import time
    if progress:
        import psutil
    started = time.perf_counter()
    total_chunks = (len(token_ids) + chunk_size - 1) // chunk_size
    combined = torch.empty(0, 3, dtype=torch.int64)
    for chunk_number, chunk_counts in enumerate(iter_pair_counts(token_ids, C, chunk_size), 1):
        combined = torch.cat([combined, chunk_counts], dim=0)
        pairs = combined[:, :2]
        counts = combined[:, 2]
        assert pairs.shape == (combined.shape[0], 2)
        assert counts.shape == (combined.shape[0],)
        unique_pairs, inverse = torch.unique(pairs, dim=0, return_inverse=True)
        U = unique_pairs.shape[0]
        assert unique_pairs.shape == (U, 2)
        assert inverse.shape == counts.shape

        # inverse maps each input row to its unique pair's group.
        totals = counts.new_zeros(U)
        assert totals.shape == (U,)
        totals.scatter_add_(0, inverse, counts)
        assert totals.shape == (U,)
        totals_column = totals.unsqueeze(1)
        assert totals_column.shape == (U, 1)
        combined = torch.cat([unique_pairs, totals_column], dim=1)
        assert combined.shape == (U, 3)
        if progress:
            processed = min(chunk_number * chunk_size, len(token_ids))
            ram = psutil.Process().memory_info().rss / 1024**3
            available = psutil.virtual_memory().available / 1024**3
            print(f'Chunk {chunk_number}/{total_chunks} | tokens {processed:,}/{len(token_ids):,} | '
                  f'pairs {U:,} | elapsed {time.perf_counter() - started:.1f}s | '
                  f'RAM {ram:.2f} GiB, available {available:.2f} GiB', flush=True)
    return combined

In [ ]:
# Small corpus sanity check before scaling up
pair_counts = combine(token_ids[:sample_size], C, chunk_size)
print(pair_counts.shape, pair_counts[:, 2].sum())

In [ ]:
class GloVe(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super().__init__()
        self.vocab_size = vocab_size
        self.embedding_dim = embedding_dim
        self.word_embeddings = nn.Embedding(vocab_size+1, embedding_dim)
        self.context_embeddings = nn.Embedding(vocab_size+1, embedding_dim)
        assert self.word_embeddings.weight.shape == (vocab_size+1, embedding_dim)
        assert self.context_embeddings.weight.shape == (vocab_size+1, embedding_dim)

        self.word_biases = nn.Embedding(vocab_size+1, 1)
        self.context_biases = nn.Embedding(vocab_size+1, 1)

        assert self.word_biases.weight.shape == (vocab_size+1, 1)
        assert self.context_biases.weight.shape == (vocab_size+1, 1)

    def forward(self, token_ids, context_ids):
        assert token_ids.shape == context_ids.shape
        targets = self.word_embeddings(token_ids)
        context = self.context_embeddings(context_ids)
        assert targets.shape == (token_ids.shape[0], self.embedding_dim)
        assert context.shape == (context_ids.shape[0], self.embedding_dim)

        target_bias = self.word_biases(token_ids)
        context_bias = self.context_biases(context_ids)
        assert target_bias.shape == (token_ids.shape[0], 1)
        assert context_bias.shape == (context_ids.shape[0], 1)

        predictions = targets * context
        predictions = predictions.sum(dim=1, keepdim=True)
        predictions += (target_bias + context_bias)
        assert predictions.shape == (token_ids.shape[0], 1)
        return predictions



In [ ]:
def glove_loss(predictions, counts, count_threshold=100, alpha=0.75):
    f_count = torch.where(counts < count_threshold, (counts/count_threshold)**alpha, 1.0)
    log_count = torch.log(counts)
    error = predictions - log_count
    loss = torch.mean(f_count*(error**2))
    return loss



In [ ]:
def prepare_data_loaders(token_ids, context_ids, counts, batch_size=32, shuffle=True):
    train_dataset = TensorDataset(token_ids, context_ids, counts)
    train_data_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=shuffle)
    return train_data_loader



In [ ]:
def train(model, pairs, counts, learning_rate, epochs, count_threshold, alpha,
          batch_size, checkpoint_path, word_to_idx, run_config):
    from pathlib import Path
    assert run_config['optimizer'] == 'SGD', 'This training loop implements SGD only'

    checkpoint_path = Path(checkpoint_path)
    if str(checkpoint_path).startswith('/content/drive/'):
        if not Path('/content/drive/MyDrive').is_dir():
            raise RuntimeError('Mount Google Drive before starting training.')
    checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
    assert pairs.shape == (counts.numel(), 2) and counts.numel() > 0
    assert bool((counts > 0).all()), 'GloVe requires positive counts'
    train_losses = []
    model = model.to(device)
    model.train()
    token_ids = pairs[:, 0]
    context_ids = pairs[:, 1]
    counts = counts.reshape(counts.shape[0], 1)
    counts = counts.float()
    train_data_loader = prepare_data_loaders(token_ids, context_ids, counts, batch_size=batch_size)
    optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
    for epoch in range(epochs):
        total_loss = 0.0
        for batch_idx, (token_ids, context_ids, counts) in enumerate(train_data_loader):
            token_ids, context_ids, counts = token_ids.to(device), context_ids.to(device), counts.to(device)
            predictions = model(token_ids, context_ids)
            loss = glove_loss(predictions, counts, count_threshold=count_threshold, alpha=alpha)
            if not torch.isfinite(loss).item():
                raise RuntimeError(f'Non-finite loss at epoch {epoch + 1}, batch {batch_idx}')
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * token_ids.shape[0]
        avg_loss = total_loss / len(train_data_loader.dataset)
        train_losses.append(avg_loss)

        # Save the completed epoch before starting the next one.
        checkpoint = {
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_losses': train_losses,
            'config': dict(run_config),
            'word_to_idx': dict(word_to_idx),
            'torch_rng_state': torch.get_rng_state(),
        }
        if device.type == 'cuda':
            checkpoint['cuda_rng_state_all'] = torch.cuda.get_rng_state_all()
        # Write a temporary file first, then replace the previous checkpoint.
        temporary_path = checkpoint_path.with_suffix('.tmp')
        torch.save(checkpoint, temporary_path)
        temporary_path.replace(checkpoint_path)
        print(f'Epoch {epoch + 1}/{epochs} | loss={avg_loss:.6f} | saved {checkpoint_path}', flush=True)
    return train_losses

In [ ]:
# Bounded benchmark: no checkpoints, no changes to your training model.
def benchmark_glove(sample_sizes=(10_000, 100_000, 300_000), steps=100, warmup=10):
    import gc
    import time
    import psutil

    if device.type not in ("cpu", "cuda"):
        raise RuntimeError("Run this benchmark on Colab (CUDA), not the local MPS kernel.")
    assert optimizer_name == 'SGD', 'This benchmark implements SGD only'
    assert steps > 0 and warmup >= 0
    process = psutil.Process()
    gib = 1024 ** 3

    def sync():
        if device.type == "cuda":
            torch.cuda.synchronize(device)

    def memory():
        return process.memory_info().rss / gib, psutil.virtual_memory().available / gib

    print(f"Device: {device}; vocab={vocab_size:,}; D={embedding_dim}; C={C}; batch={batch_size}")
    if device.type == "cuda":
        print(f"GPU: {torch.cuda.get_device_name(device)}")
    rss, available = memory()
    print(f"Initial process RAM: {rss:.2f} GiB; system available: {available:.2f} GiB")
    results = {"preprocessing": []}
    bench_counts = None
    for size in sorted({min(size, len(token_ids)) for size in sample_sizes}):
        if size < 2:
            continue
        # Release previous sample before preparing the next one.
        bench_counts = None
        gc.collect()
        started = time.perf_counter()
        bench_counts = combine(token_ids[:size], C, chunk_size)
        elapsed = time.perf_counter() - started
        rss, available = memory()
        row = dict(tokens=size, unique_pairs=len(bench_counts), seconds=elapsed,
                   process_ram_gib=rss, available_ram_gib=available)
        results["preprocessing"].append(row)
        print(f"Counts: {size:,} tokens -> {len(bench_counts):,} pairs | {elapsed:.2f}s | "
              f"RAM now {rss:.2f} GiB; available {available:.2f} GiB", flush=True)
        # Stop scaling samples if one is already slow or RAM is tight.
        if elapsed > benchmark_time_limit_seconds or available < benchmark_min_available_ram_gib:
            print("Stopping preprocessing benchmark early: time/RAM limit reached.")
            break
    if bench_counts is None or len(bench_counts) == 0:
        raise ValueError("Need at least one observed pair for the benchmark.")

    # Separate model, and restore RNG state so this test does not change training initialization.
    cuda_devices = [torch.cuda.current_device()] if device.type == "cuda" else []
    with torch.random.fork_rng(devices=cuda_devices):
        bench_model = GloVe(vocab_size, embedding_dim).to(device)
        bench_model.train()
        bench_optimizer = torch.optim.SGD(bench_model.parameters(), lr=learning_rate)
        loader = prepare_data_loaders(
            bench_counts[:, 0], bench_counts[:, 1], bench_counts[:, 2:3].float(),
            batch_size=batch_size,
        )

        def batches():
            while True:
                yield from loader

        batches_iter = batches()

        def step():
            target, context, count = next(batches_iter)
            target, context, count = target.to(device), context.to(device), count.to(device)
            prediction = bench_model(target, context)
            loss = glove_loss(prediction, count, count_threshold, alpha)
            if not torch.isfinite(loss).item():
                raise RuntimeError("Non-finite benchmark loss.")
            bench_optimizer.zero_grad()
            loss.backward()
            bench_optimizer.step()
            return loss.item(), target.shape[0]

        for _ in range(warmup):
            step()
        sync()
        started = time.perf_counter()
        examples = 0
        completed = 0
        for _ in range(steps):
            last_loss, size = step()
            examples += size
            completed += 1
            if time.perf_counter() - started > benchmark_time_limit_seconds:
                break
        sync()
        elapsed = time.perf_counter() - started
        results["training"] = dict(steps=completed, seconds=elapsed,
                                   pairs_per_second=examples / elapsed, last_loss=last_loss)
        print(f"Training: {completed} steps in {elapsed:.2f}s | "
              f"{examples / elapsed:,.0f} pairs/s | last loss={last_loss:.4f}")
        if device.type == "cuda":
            print(f"GPU allocated now: {torch.cuda.memory_allocated(device) / gib:.2f} GiB "
                  "(includes any other live models; not peak memory)")
        del bench_model, bench_optimizer, batches_iter, loader
    gc.collect()
    print("No checkpoint written. RAM readings are snapshots, not peaks.")
    print("Full-corpus time is NOT a linear extrapolation: combine repeatedly merges growing counts.")
    return results

benchmark_results = benchmark_glove(
    sample_sizes=benchmark_sample_sizes, steps=benchmark_steps, warmup=benchmark_warmup,
)

In [ ]:
benchmark_results = benchmark_glove(
    sample_sizes=benchmark_large_sample_sizes, steps=benchmark_steps, warmup=benchmark_warmup,
)



In [ ]:
# Full-corpus preprocessing only. Keep the final training cell OFF.
# RUN_FULL_COUNTS and full_counts_path are configured in cell 5.

def prepare_full_counts(token_ids, C, chunk_size, word_to_idx, save_path):
    from pathlib import Path
    import time

    save_path = Path(save_path)
    if str(save_path).startswith('/content/drive/') and not Path('/content/drive/MyDrive').is_dir():
        raise RuntimeError('Mount Google Drive first.')
    if save_path.exists():
        raise FileExistsError(f'{save_path} already exists. Load it or choose a new path; do not overwrite.')
    save_path.parent.mkdir(parents=True, exist_ok=True)
    started = time.perf_counter()
    print(f'Preparing full counts for {len(token_ids):,} tokens; C={C}. No training.', flush=True)
    result = combine(token_ids, C, chunk_size, progress=True)
    # Raw, symmetric unit counts: both directions for each valid distance.
    radius = min(C, max(0, len(token_ids) - 1))
    expected_total = 2 * (radius * len(token_ids) - radius * (radius + 1) // 2)
    assert int(result[:, 2].sum()) == expected_total
    metadata = dict(num_tokens=len(token_ids), C=C, chunk_size=chunk_size,
                    weighting='unit', vocab_size=len(word_to_idx))
    temporary_path = save_path.with_suffix('.tmp')
    print(f'Saving {len(result):,} unique pairs to {save_path} ...', flush=True)
    torch.save(dict(pair_counts=result, word_to_idx=word_to_idx, metadata=metadata),
               temporary_path)
    temporary_path.replace(save_path)
    print(f'Done: {len(result):,} pairs | total count={expected_total:,} | '
          f'{time.perf_counter() - started:.1f}s including save | '
          f'file={save_path.stat().st_size / 1024**2:.1f} MiB', flush=True)
    return result, metadata

if RUN_FULL_COUNTS:
    from pathlib import Path
    pair_counts, counts_metadata = prepare_full_counts(
        token_ids, C, chunk_size, word_to_idx, full_counts_path,
    )
    # Keep checkpoint metadata consistent with the data now used by the training cell.
    run_config.update(sample_size=len(token_ids), C=C, chunk_size=chunk_size,
                      counts_path=str(full_counts_path), weighting='unit')
else:
    print('Full preprocessing skipped. Set RUN_FULL_COUNTS=True to prepare and save counts.')

In [ ]:
# Enable only after reviewing benchmark results and the training config.
# RUN_TRAINING is configured in cell 5.

if RUN_TRAINING:
    model = GloVe(vocab_size, embedding_dim)
    pairs = pair_counts[:, :2]
    counts = pair_counts[:, 2]
    train_losses = train(model, pairs, counts, learning_rate, epochs, count_threshold, alpha,
                         batch_size, checkpoint_path, word_to_idx, run_config)
    print(train_losses)
else:
    print("Training skipped. Review benchmark results first.")

In [ ]:
# Evaluation
def get_embeddings(model, add_context_embeddings=False):
    model.eval()
    with torch.no_grad():
        final_embedding = model.word_embeddings.weight + model.context_embeddings.weight if add_context_embeddings else model.word_embeddings.weight
    assert final_embedding.shape == (model.vocab_size + 1, model.embedding_dim)
    normalized_embedding = torch.nn.functional.normalize(final_embedding, p=2, dim=1)
    return normalized_embedding
    
def load_model(model_path, device='cuda'):
    checkpoint = torch.load(model_path, map_location=device, weights_only=True)
    model = GloVe(checkpoint["config"]["vocab_size"], checkpoint["config"]["embedding_dim"])
    model.load_state_dict(checkpoint["model_state_dict"])
    return model, checkpoint

def get_topk_matches(word, normalized_embedding, checkpoint ,topk=10):
    word_to_idx = checkpoint["word_to_idx"]
    idx_to_word = {value:key for key, value in word_to_idx.items()}
    word_idx = word_to_idx[word]
    word_embedding = normalized_embedding[word_idx]
    #normalized_embedding[word_idx] = -float("inf")
    scores = normalized_embedding @ word_embedding
    scores[0] = -float("inf")
    scores[word_idx] = -float("inf")
    assert scores.shape == (vocab_size+1,)
    topk_scores, indices = torch.topk(scores, k=topk)
    return [idx_to_word.get(ind.item(), None) for ind in indices], topk_scores.tolist()

In [ ]:
model, checkpoint = load_model(trained_checkpoint_path)
normalized_embedding = get_embeddings(model)

In [ ]:
topk_similar_words = get_topk_matches("ice", normalized_embedding, checkpoint)

In [ ]:
topk_similar_words

In [ ]:
torch.manual_seed(42)
initial_model = GloVe(vocab_size, embedding_dim)
word_embedding = initial_model.word_embeddings.weight
initial_ice_embedding = word_embedding[word_to_idx["ice"]]

current_word_embedding = model.word_embeddings.weight
current_ice_embedding = current_word_embedding[checkpoint["word_to_idx"]["ice"]]
print(
    f"Initial length: {torch.linalg.vector_norm(initial_ice_embedding).item():.4f}\n"
    f"Final length:   {torch.linalg.vector_norm(current_ice_embedding).item():.4f}"
)

print(
    f"Cosine Similarity: {torch.nn.functional.cosine_similarity(initial_ice_embedding, current_ice_embedding, dim=0)}"
)

In [ ]:
# Compare the same full-corpus ice-target subset before and after training.
saved_counts = torch.load(
    checkpoint["config"]["counts_path"], map_location="cpu", weights_only=True,
)
assert saved_counts["word_to_idx"] == checkpoint["word_to_idx"]
ice_id = checkpoint["word_to_idx"]["ice"]
ice_pairs = saved_counts["pair_counts"]
ice_pairs = ice_pairs[ice_pairs[:, 0] == ice_id]
assert ice_pairs.ndim == 2 and ice_pairs.shape[1] == 3 and ice_pairs.shape[0] > 0
del saved_counts

def ice_subset_loss(candidate_model, topk=5):
    candidate_device = next(candidate_model.parameters()).device
    _, top_indices = torch.topk(ice_pairs[:, -1], k=topk)
    target_ids = ice_pairs[top_indices, 0].to(candidate_device)
    context_ids = ice_pairs[top_indices, 1].to(candidate_device)
    counts = ice_pairs[top_indices, 2:3].float().to(candidate_device)
    with torch.no_grad():
        predictions = candidate_model(target_ids, context_ids)
        assert predictions.shape == counts.shape == (len(ice_pairs[top_indices]), 1)
        return glove_loss(
            predictions, counts,
            count_threshold=checkpoint["config"]["count_threshold"],
            alpha=checkpoint["config"]["alpha"],
        ).item(), predictions, counts
initial_loss, initial_predictions, counts = ice_subset_loss(initial_model)
trained_loss, trained_predictions, counts = ice_subset_loss(model)
print(f"Initial loss: {initial_loss:.4f}\nTrained loss: {trained_loss:.4f}")
print(f"Initial Prediction: {initial_predictions}\nFinal Prediction: {trained_predictions}\nCounts: {torch.log(counts)}")